# AudioRestore: Fine-tuning SonicMaster for Codec Audio Restoration

**CS 614 Final Project** — Fine-tune [SonicMaster](https://github.com/AMAAI-Lab/SonicMaster) (ICLR 2026, 0.9B params) on codec-degraded audio.

**Runs on**: Google Colab (A100/T4) **or** local machine (AMD ROCm / NVIDIA CUDA).

## Workflow
1. Detect environment (Colab vs local)
2. Install dependencies (Colab only; local: use conda env)
3. Set project paths (Drive on Colab, `Project/` on local)
4. Download pretrained SonicMaster checkpoint
5. Prepare codec-degraded dataset
6. Pre-encode VAE latents
7. Fine-tune
8. Plot loss curves

In [1]:
# Cell 0: Detect environment (Colab vs local)
import os
import sys

try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    PROJECT_ROOT = '/content/drive/MyDrive/CS614_Project'
    SONICMASTER_DIR = '/content/sonicmaster'
    CONFIGS_DIR = '/content'
else:
    # Local: infer Project/ from cwd (run notebook from CS614/ or Project/)
    cwd = os.getcwd()
    if os.path.basename(cwd) == 'Project':
        PROJECT_ROOT = cwd
    elif os.path.isdir(os.path.join(cwd, 'Project')):
        PROJECT_ROOT = os.path.join(cwd, 'Project')
    else:
        PROJECT_ROOT = cwd  # assume cwd is Project/
    SONICMASTER_DIR = os.path.join(PROJECT_ROOT, 'sonicmaster')
    CONFIGS_DIR = os.path.join(PROJECT_ROOT, 'configs')

if not IS_COLAB:
    os.makedirs(PROJECT_ROOT, exist_ok=True)
    os.makedirs(os.path.join(PROJECT_ROOT, 'checkpoints'), exist_ok=True)
    os.makedirs(os.path.join(PROJECT_ROOT, 'data'), exist_ok=True)
    os.makedirs(os.path.join(PROJECT_ROOT, 'outputs'), exist_ok=True)

print(f"Environment: {'Colab' if IS_COLAB else 'Local'}")
print(f"Project root: {PROJECT_ROOT}")
print(f"SonicMaster dir: {SONICMASTER_DIR}")

Environment: Local
Project root: /home/kagamirudo/CS614/Project
SonicMaster dir: /home/kagamirudo/CS614/Project/sonicmaster


In [2]:
# Cell 1: Install dependencies (Colab only; local: use conda env with deps already installed)
if IS_COLAB:
    !pip install torch==2.4.0 torchaudio==2.4.0 torchvision==0.19.0 -q
    !pip install diffusers==0.30.0 transformers==4.44.0 accelerate==0.34.2 -q
    !pip install safetensors datasets librosa soundfile tqdm pandas pyyaml -q
    !pip install huggingface_hub -q
    print("Dependencies installed.")
else:
    print("Local run: using existing conda/env. Ensure: torch, torchaudio, diffusers, transformers, accelerate, safetensors, datasets, librosa, soundfile, tqdm, pandas, pyyaml, huggingface_hub")

Local run: using existing conda/env. Ensure: torch, torchaudio, diffusers, transformers, accelerate, safetensors, datasets, librosa, soundfile, tqdm, pandas, pyyaml, huggingface_hub


In [3]:
# Cell 2: Mount Google Drive (Colab only)
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/CS614_Project'
    os.makedirs(PROJECT_ROOT, exist_ok=True)
    os.makedirs(f'{PROJECT_ROOT}/checkpoints', exist_ok=True)
    os.makedirs(f'{PROJECT_ROOT}/data', exist_ok=True)
    os.makedirs(f'{PROJECT_ROOT}/outputs', exist_ok=True)
    print(f'Drive mounted. Project root: {PROJECT_ROOT}')
else:
    print(f'Local run. Project root: {PROJECT_ROOT}')

Local run. Project root: /home/kagamirudo/CS614/Project


In [4]:
# Cell 3: Set up SonicMaster code
# Colab: clone to /content/sonicmaster
# Local: use Project/sonicmaster (already cloned)
if IS_COLAB:
    if not os.path.exists('/content/sonicmaster'):
        get_ipython().system('git clone https://github.com/AMAAI-Lab/SonicMaster.git /content/sonicmaster')
    sys.path.insert(0, '/content/sonicmaster')
    SONICMASTER_DIR = '/content/sonicmaster'
else:
    if not os.path.isdir(SONICMASTER_DIR):
        get_ipython().system(f'git clone https://github.com/AMAAI-Lab/SonicMaster.git {SONICMASTER_DIR}')
    sys.path.insert(0, SONICMASTER_DIR)
sys.path.insert(0, PROJECT_ROOT)
print(f'SonicMaster ready: {SONICMASTER_DIR}')

SonicMaster ready: /home/kagamirudo/CS614/Project/sonicmaster


In [5]:
# Cell 4: Authenticate with HuggingFace (needed for Stable Audio VAE)
# If login() widget fails (missing ipywidgets), fall back to token-based login
import os
from huggingface_hub import login

token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
if token:
    login(token=token)
    print("Logged in via HF_TOKEN env var.")
else:
    try:
        login()
    except ImportError:
        print("Widget login unavailable. Use one of:")
        print("  1) Set env var: export HF_TOKEN=hf_...")
        print("  2) Run in terminal: huggingface-cli login")
        print("  3) pip install ipywidgets, restart kernel, re-run this cell")

Widget login unavailable. Use one of:
  1) Set env var: export HF_TOKEN=hf_...
  2) Run in terminal: huggingface-cli login
  3) pip install ipywidgets, restart kernel, re-run this cell


/opt/conda-data/envs/apollo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Cell 5: Download pretrained SonicMaster checkpoint
CKPT_PATH = os.path.join(PROJECT_ROOT, 'checkpoints', 'model.safetensors')

if not os.path.exists(CKPT_PATH):
    from huggingface_hub import hf_hub_download
    print('Downloading SonicMaster checkpoint (~3.6 GB)...')
    hf_hub_download(
        repo_id='amaai-lab/SonicMaster',
        filename='model.safetensors',
        local_dir=os.path.join(PROJECT_ROOT, 'checkpoints'),
    )
    print('Done!')
else:
    print(f'Checkpoint already exists: {CKPT_PATH}')

Checkpoint already exists: /home/kagamirudo/CS614/Project/checkpoints/model.safetensors


In [7]:
# Cell 6: Locate clean audio for dataset preparation
# Put clean WAV/FLAC files in Project/data/clean/ (or Colab: Drive/CS614_Project/data/clean/)

DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
CLEAN_DIR = os.path.join(DATA_DIR, 'clean')
os.makedirs(CLEAN_DIR, exist_ok=True)

if len(os.listdir(CLEAN_DIR)) == 0:
    import shutil, glob
    _ws_root = os.path.dirname(PROJECT_ROOT)
    _fallback = (
        glob.glob(os.path.join(_ws_root, 'Applications', 'asserts', '*.wav')) +
        glob.glob(os.path.join(_ws_root, 'Applications', 'asserts', '*.mp3')) +
        glob.glob(os.path.join(_ws_root, 'Applications', '*.wav'))
    )
    if _fallback:
        print(f'No audio in {CLEAN_DIR}. Copying {len(_fallback)} files from Applications/ for testing...')
        for _src in _fallback:
            shutil.copy2(_src, CLEAN_DIR)
    else:
        print('Upload clean WAV/FLAC files to:', CLEAN_DIR)
        print('Or download MUSDB18-HQ test set:')
        print('  wget https://zenodo.org/records/3338373/files/musdb18hq.zip')
        print('  (extract and copy test/ stems to', CLEAN_DIR, ')')

if len(os.listdir(CLEAN_DIR)) > 0:
    n = len([f for f in os.listdir(CLEAN_DIR) if f.endswith(('.wav', '.flac', '.mp3'))])
    print(f'Found {n} audio files in {CLEAN_DIR}')

Found 113 audio files in /home/kagamirudo/CS614/Project/data/clean


In [ ]:
# Cell 7: Generate codec-degraded dataset
import json, random, os
import numpy as np
import torch
import torchaudio
from torchaudio.io import AudioEffector, CodecConfig
import soundfile as sf
from pathlib import Path
from tqdm import tqdm

SAMPLE_RATE = 44100
CLIP_DURATION = 30
CLIP_SAMPLES = SAMPLE_RATE * CLIP_DURATION
NUM_CLIPS = 1000
VAL_RATIO = 0.1

MP3_BITRATES = [24000, 32000, 48000, 64000, 96000, 128000]
OGG_QUALITIES = [-1, 0, 1, 2, 3, 5]

PROMPTS_MP3 = [
    "restore audio compressed at {br}kbps MP3",
    "remove MP3 compression artifacts from {br}kbps audio",
    "enhance low bitrate MP3 audio to high fidelity",
    "fix MP3 compression artifacts and restore high frequencies",
]
PROMPTS_OGG = [
    "restore audio compressed with OGG Vorbis at quality {q}",
    "remove OGG compression artifacts from low quality audio",
    "enhance OGG-compressed audio to high fidelity",
]
ALT_PROMPTS = [
    "make this audio sound better",
    "improve the sound quality",
    "restore the original audio quality",
    "enhance this compressed audio",
]

def find_audio(d):
    exts = {'.wav', '.flac', '.mp3', '.ogg'}
    return sorted([os.path.join(r,f) for r,_,fs in os.walk(d) for f in fs if Path(f).suffix.lower() in exts])

def load_prep(p):
    w, sr = torchaudio.load(p)
    if sr != SAMPLE_RATE: w = torchaudio.functional.resample(w, sr, SAMPLE_RATE)
    if w.shape[0]==1: w = w.repeat(2,1)
    elif w.shape[0]>2: w = w[:2]
    return w

def rand_clip(w):
    T = w.shape[1]
    if T >= CLIP_SAMPLES:
        s = random.randint(0, T-CLIP_SAMPLES)
        return w[:, s:s+CLIP_SAMPLES]
    return torch.nn.functional.pad(w, (0, CLIP_SAMPLES-T))

random.seed(42)
files = find_audio(CLEAN_DIR)
assert files, f'No audio files in {CLEAN_DIR}'
print(f'{len(files)} source files — loading into memory...')

audio_cache = {}
for f in tqdm(files, desc='Caching audio'):
    audio_cache[f] = load_prep(f)
print(f'Cached {len(audio_cache)} files in RAM')

mp3_effectors = {br: AudioEffector(format='mp3', encoder='libmp3lame', codec_config=CodecConfig(bit_rate=br)) for br in MP3_BITRATES}
ogg_effectors = {q: AudioEffector(format='ogg', encoder='libvorbis', codec_config=CodecConfig(qscale=q)) for q in OGG_QUALITIES}

clean_out = f'{DATA_DIR}/clean_clips'; os.makedirs(clean_out, exist_ok=True)
deg_out = f'{DATA_DIR}/degraded_clips'; os.makedirs(deg_out, exist_ok=True)

records = []
for i in tqdm(range(NUM_CLIPS), desc='Generating clips'):
    try:
        clip = rand_clip(audio_cache[random.choice(files)])
        peak = clip.abs().max()
        if peak > 0: clip = clip / peak * 0.95
        
        if random.random() < 0.7:
            br = random.choice(MP3_BITRATES)
            deg = mp3_effectors[br].apply(clip.T, SAMPLE_RATE).T
            prompt = random.choice(PROMPTS_MP3).format(br=br//1000)
        else:
            q = random.choice(OGG_QUALITIES)
            deg = ogg_effectors[q].apply(clip.T, SAMPLE_RATE).T
            prompt = random.choice(PROMPTS_OGG).format(q=q)
        
        if deg.shape[1] != clip.shape[1]:
            deg = deg[:, :clip.shape[1]] if deg.shape[1] > clip.shape[1] else torch.nn.functional.pad(deg, (0, clip.shape[1]-deg.shape[1]))
        
        cn = f'clip_{i:05d}_clean.flac'
        dn = f'clip_{i:05d}_degraded.flac'
        sf.write(f'{clean_out}/{cn}', clip.numpy().T, SAMPLE_RATE)
        sf.write(f'{deg_out}/{dn}', deg.numpy().T, SAMPLE_RATE)
        records.append({'prompt': prompt, 'alt_prompt': random.choice(ALT_PROMPTS),
                        'original_location': f'{clean_out}/{cn}', 'location': f'{deg_out}/{dn}', 'duration': 30})
    except Exception as e:
        print(f'Skip {i}: {e}')

del audio_cache

random.shuffle(records)
nv = max(1, int(len(records)*VAL_RATIO))
for path, recs in [(f'{DATA_DIR}/trainset.jsonl', records[nv:]), (f'{DATA_DIR}/valset.jsonl', records[:nv])]:
    with open(path,'w') as f:
        for r in recs: f.write(json.dumps(r)+'\n')
print(f'{len(records)-nv} train + {nv} val clips generated.')

113 source files


 80%|████████  | 800/1000 [07:42<01:55,  1.73it/s]

In [ ]:
# Cell 8: Pre-encode VAE latents
# Auto-selects batch size: 1 for <=10 GB VRAM (local), 4+ for Colab A100/T4
import os, torch, torchaudio, json
from tqdm import tqdm
if not IS_COLAB and torch.cuda.is_available() and hasattr(torch.version, 'hip'):
    os.environ.setdefault('PYTORCH_HIP_ALLOC_CONF', 'expandable_segments:True')

from diffusers import AutoencoderOobleck

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device != 'cpu':
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    use_fp16 = gpu_mem_gb < 16
    BATCH_SIZE = 1 if gpu_mem_gb < 16 else (4 if gpu_mem_gb < 24 else 8)
else:
    gpu_mem_gb = 0
    use_fp16 = False
    BATCH_SIZE = 1

vae_dtype = torch.float16 if use_fp16 else torch.float32
vae = AutoencoderOobleck.from_pretrained('stabilityai/stable-audio-open-1.0', subfolder='vae', torch_dtype=vae_dtype).to(device)
vae.eval()
_ = vae.requires_grad_(False)
print(f'VAE encoding: device={device}, VRAM={gpu_mem_gb:.1f} GB, dtype={vae_dtype}, batch_size={BATCH_SIZE}')

def pad_wav(w, n):
    return w[:n] if w.shape[0]>=n else torch.cat([w, torch.zeros(n-w.shape[0])])

def load_audio_for_vae(path):
    w, sr = torchaudio.load(path)
    if sr != 44100: w = torchaudio.functional.resample(w, sr, 44100)
    if w.shape[0]==1: w = w.repeat(2,1)
    elif w.shape[0]>2: w = w[:2]
    t = 44100*30
    return torch.stack([pad_wav(w[0],t), pad_wav(w[1],t)])

def encode_batch(wavs):
    """Encode a list of [2, T] waveforms, return list of [seq_len, channels] latents (float32)."""
    with torch.no_grad():
        inp = torch.stack(wavs).to(device=device, dtype=vae_dtype)
        latents = vae.encode(inp).latent_dist.mode().transpose(1, 2).float().cpu()
        del inp
        torch.cuda.empty_cache()
        return [latents[i] for i in range(latents.shape[0])]

skipped = 0
for jsonl_path in [f'{DATA_DIR}/trainset.jsonl', f'{DATA_DIR}/valset.jsonl']:
    with open(jsonl_path) as f:
        records = [json.loads(l) for l in f if l.strip()]
    
    clean_pt_dir = f'{DATA_DIR}/clean_pt'; os.makedirs(clean_pt_dir, exist_ok=True)
    deg_pt_dir = f'{DATA_DIR}/degraded_pt'; os.makedirs(deg_pt_dir, exist_ok=True)
    
    updated = []
    batch_c, batch_d, batch_r = [], [], []
    
    for rec in tqdm(records, desc=f'Encoding {os.path.basename(jsonl_path)}'):
        try:
            batch_c.append(load_audio_for_vae(rec['original_location']))
            batch_d.append(load_audio_for_vae(rec['location']))
            batch_r.append(rec)
            
            if len(batch_c) >= BATCH_SIZE:
                cl = encode_batch(batch_c)
                dl = encode_batch(batch_d)
                for j, r in enumerate(batch_r):
                    cn = os.path.basename(r['original_location']).rsplit('.',1)[0]+'.pt'
                    dn = os.path.basename(r['location']).rsplit('.',1)[0]+'.pt'
                    cp = f'{clean_pt_dir}/{cn}'; dp = f'{deg_pt_dir}/{dn}'
                    torch.save(cl[j], cp); torch.save(dl[j], dp)
                    updated.append({**r, 'original_location': cp, 'location': dp})
                batch_c, batch_d, batch_r = [], [], []
        except Exception as e:
            skipped += 1
            if skipped <= 3:
                print(f'  Skip: {e}')
            elif skipped == 4:
                print(f'  (suppressing further skip messages...)')
            batch_c, batch_d, batch_r = [], [], []
            torch.cuda.empty_cache()
    
    if batch_c:
        try:
            cl = encode_batch(batch_c)
            dl = encode_batch(batch_d)
            for j, r in enumerate(batch_r):
                cn = os.path.basename(r['original_location']).rsplit('.',1)[0]+'.pt'
                dn = os.path.basename(r['location']).rsplit('.',1)[0]+'.pt'
                cp = f'{clean_pt_dir}/{cn}'; dp = f'{deg_pt_dir}/{dn}'
                torch.save(cl[j], cp); torch.save(dl[j], dp)
                updated.append({**r, 'original_location': cp, 'location': dp})
        except Exception as e:
            print(f'  Skip final batch: {e}')
            torch.cuda.empty_cache()
    
    pt_jsonl = jsonl_path.replace('.jsonl', '_pt.jsonl')
    with open(pt_jsonl, 'w') as f:
        for r in updated: f.write(json.dumps(r)+'\n')
    print(f'  {len(updated)} encoded, {skipped} skipped -> {pt_jsonl}')

del vae
torch.cuda.empty_cache()
print('\nLatent encoding complete!')

VAE encoding: device=cuda, VRAM=8.0 GB, dtype=torch.float16, batch_size=1


Encoding trainset.jsonl:   0%|          | 1/900 [00:30<7:39:06, 30.64s/it]

In [ ]:
# Cell 9: Write fine-tuning config pointing to our data
import yaml

finetune_config = {
    'paths': {
        'train_file': os.path.join(DATA_DIR, 'trainset_pt.jsonl'),
        'val_file': os.path.join(DATA_DIR, 'valset_pt.jsonl'),
        'test_file': os.path.join(DATA_DIR, 'valset_pt.jsonl'),
        'infer_file': os.path.join(DATA_DIR, 'valset_pt.jsonl'),
        'resume_from_checkpoint': '',
        'output_dir': os.path.join(PROJECT_ROOT, 'outputs', 'finetune_codec'),
    },
    'training': {
        'per_device_batch_size': 2,
        'learning_rate': 1e-5,
        'gradient_accumulation_steps': 4,
        'num_train_epochs': 30,
        'num_warmup_steps': 200,
        'max_audio_duration': 30,
    },
    'model': {
        'num_layers': 6,
        'num_single_layers': 18,
        'in_channels': 64,
        'attention_head_dim': 128,
        'joint_attention_dim': 1024,
        'num_attention_heads': 8,
        'audio_seq_len': 645,
        'max_duration': 30,
        'uncondition': False,
        'text_encoder_name': 'google/flan-t5-large',
    },
}

config_path = os.path.join(CONFIGS_DIR, 'finetune_codec.yaml')
os.makedirs(CONFIGS_DIR, exist_ok=True)
with open(config_path, 'w') as f:
    yaml.dump(finetune_config, f, default_flow_style=False)
print(f'Config written to {config_path}')

Config written to /home/kagamirudo/CS614/Project/configs/finetune_codec.yaml


In [ ]:
# Cell 10: Write accelerator config for single GPU
import json

accel_config = {
    'compute_environment': 'LOCAL_MACHINE',
    'distributed_type': 'NO',
    'main_process_port': 29512,
    'downcast_bf16': False,
    'machine_rank': 0,
    'main_training_function': 'main',
    'mixed_precision': 'fp16',
    'num_machines': 1,
    'num_processes': 1,
    'rdzv_backend': 'static',
    'same_network': True,
    'tpu_use_cluster': False,
    'tpu_use_sudo': False,
    'use_cpu': False,
}

accel_path = os.path.join(CONFIGS_DIR, 'accelerator_single_gpu.yaml')
with open(accel_path, 'w') as f:
    json.dump(accel_config, f, indent=2)
print(f'Accelerator config written to {accel_path}')

Accelerator config written to /home/kagamirudo/CS614/Project/configs/accelerator_single_gpu.yaml


In [ ]:
# Cell 11: Training script path
# We use SonicMaster's train_ptload_inference.py (same for Colab and local)
TRAIN_SCRIPT = os.path.join(SONICMASTER_DIR, 'train_ptload_inference.py')
assert os.path.exists(TRAIN_SCRIPT), f'Training script not found: {TRAIN_SCRIPT}'
print(f'Training script: {TRAIN_SCRIPT}')

Training script: /home/kagamirudo/CS614/Project/sonicmaster/train_ptload_inference.py


In [ ]:
# Cell 12: Launch fine-tuning!
# Colab: run from /content. Local: run from Project/ (cwd)
# #region agent log
import json as _json, time as _time
with open('/home/kagamirudo/CS614/.cursor/debug-166830.log', 'a') as _f:
    _f.write(_json.dumps({'sessionId':'166830','location':'train_colab.ipynb:cell13','message':'Training launch','data':{'wandb_mode':'disabled'},'timestamp':int(_time.time()*1000),'hypothesisId':'C'})+'\n')
# #endregion
os.environ['WANDB_MODE'] = 'disabled'
cmd = f"accelerate launch --config_file {accel_path} {TRAIN_SCRIPT} --config {config_path} --load_from_checkpoint {CKPT_PATH} --learning_rate 1e-5 --lr_scheduler_type cosine --save_every 5 --seed 42"
get_ipython().system(cmd)

/opt/conda-data/envs/apollo/lib/python3.11/site-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
03/18/2026 01:08:11 - INFO - __main__ - Distributed environment: DistributedType.NO
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda

Mixed precision type: fp16

Traceback (most recent call last):
  File "/home/kagamirudo/CS614/Project/sonicmaster/train_ptload_inference.py", line 780, in <module>
    main()
  File "/home/kagamirudo/CS614/Project/sonicmaster/train_ptload_inference.py", line 284, in main
    raw_datasets = load_dataset(extension, data_files=data_files)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda-data/envs/apollo/lib/python3.11/site-packages/datasets/load.py", line 1708, in load_dataset
    builder_instance.download_and_prepare(
  File "/opt/conda-d

In [ ]:
# Cell 13: Plot training loss curves
import matplotlib.pyplot as plt
import json

output_dir = os.path.join(PROJECT_ROOT, 'outputs', 'finetune_codec')
summary_file = os.path.join(output_dir, 'summary.jsonl')

epochs, train_losses, val_losses = [], [], []
if not os.path.exists(summary_file):
    print(f'No summary file yet: {summary_file}. Run training first.')
else:
    with open(summary_file) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                d = json.loads(line)
                if 'epoch' in d and 'epoch/train_loss' in d:
                    epochs.append(d['epoch'])
                    train_losses.append(d['epoch/train_loss'])
                    val_losses.append(d['epoch/val_loss'])
            except Exception:
                pass

if epochs:
    fig, ax = plt.subplots(1, 1, figsize=(10, 5))
    ax.plot(epochs, train_losses, label='Train Loss', marker='o', markersize=3)
    ax.plot(epochs, val_losses, label='Val Loss', marker='s', markersize=3)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_title('SonicMaster Fine-tuning: Codec Restoration')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, 'loss_curves.png'), dpi=150)
    plt.show()
    print(f'Best val loss: {min(val_losses):.4f} at epoch {epochs[val_losses.index(min(val_losses))]}')